# 📍 Station Proposal & AFIR Gap Analysis
### Iberdrola Datathon March 2026 — Objective 1 & 2

---

This notebook identifies **where new HPC charging stations must be placed** on Spain's interurban road network and **which locations face grid capacity constraints**. It produces all three mandatory output datasets.

| Output | File | Description |
|---|---|---|
| File 1 | `File 1.csv` | Global KPI scorecard (1 row) |
| File 2 | `File 2.csv` | All proposed charging stations |
| File 3 | `File 3.csv` | Friction points (Moderate + Congested only) |
| BI Map | `File_BI_visualization.html` | Self-contained interactive map |

### Methodology Overview

1. **Load grid capacity data** from all three Spanish distributors (i-DE, Endesa, Viesgo) — CNMC 2026.
2. **Load existing EV charger baseline** from the DGT National Access Point (DATEX II XML).
3. **Define TEN-T corridors** as sequences of GPS waypoints covering Spain's 12 strategic interurban axes.
4. **Sample candidate sites every 60 km** along each corridor — the AFIR Regulation 2023/1804 minimum spacing for HPC on TEN-T Core corridors.
5. **AFIR gap analysis**: retain candidates where no existing HPC charger (≥50 kW) is within 30 km.
6. **Size each station** using observed corridor traffic and projected EV fleet share (2027).
7. **Assign grid_status** via nearest-substation Haversine lookup; classify by available capacity MW.
8. **Generate File 1 / File 2 / File 3** and the BI visualisation map.

### Regulatory Reference
- **AFIR Regulation 2023/1804** (EU): HPC chargers mandatory every **60 km** on TEN-T Core corridors, **100 km** on Comprehensive corridors — deadline Dec 2025.
- **150 kW per charger** — fixed for all teams per datathon rules.

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import io, math, time, warnings, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import requests

# ── Geospatial ────────────────────────────────────────────────────────────────
import geopandas as gpd
import pyproj
from scipy.spatial import KDTree
from shapely.geometry import LineString, Point

# ── Visualisation ─────────────────────────────────────────────────────────────
import folium
from folium.plugins import MarkerCluster
from IPython.display import display, IFrame

warnings.filterwarnings('ignore')

from pathlib import Path
OUT  = Path('outputs')
DATA = Path('Data_Supply')
OUT.mkdir(exist_ok=True)

# ── Global constants (all assumptions documented here) ────────────────────────
CRS_GEO    = 'EPSG:4326'    # WGS84 — all file outputs
CRS_METRIC = 'EPSG:25830'   # ETRS89/UTM30N — distance calculations

CHARGER_KW          = 150   # Fixed by datathon rules (kW per charger)
MIN_CHARGERS        = 2     # AFIR minimum per site
MAX_CHARGERS        = 12    # Practical cap (space + demand balance)

# AFIR gap rule: TEN-T Core = 60 km max spacing
AFIR_SPACING_KM     = 60
# Gap threshold: no existing HPC charger within this radius → propose new station
GAP_RADIUS_KM       = 30   # = half of AFIR spacing
# Minimum power to count as HPC (50 kW, stored as Watts in DGT DATEX II feed)
HPC_MIN_POWER_W     = 50_000

# Demand model (all parameters from cited sources — see Section 5 markdown)
EV_SHARE_2027       = 0.038   # EV fleet / total fleet 2027 (DGT + Notebook 2.1)
STOP_PROBABILITY    = 0.15    # P(EV stops to charge | interurban trip) — MITMA 2023
SESSION_DURATION_H  = 20/60  # 20 min @ 150 kW = 50 kWh — McKinsey EV 2023
OPERATIONAL_HOURS   = 16     # 06:00–22:00 operational window
UTILIZATION_TARGET  = 0.75   # McKinsey EV Infrastructure 2023 target

# Grid status thresholds (available MW at nearest substation)
# Sufficient: ≥5 MW → handles ≥33 chargers at 150 kW without reinforcement
# Moderate:  1–5 MW → viable for standard station; may need planning coordination
# Congested: <1 MW  → grid reinforcement required before connection
GRID_SUFFICIENT_MW  = 5.0
GRID_MODERATE_MW    = 1.0

SPAIN_CENTER = [40.2, -3.7]
print('Setup complete — all constants loaded')

Setup complete — all constants loaded


---
## 1. Grid Distributor Data (i-DE, Endesa, Viesgo)

Loads CNMC capacity-access datasets published April 2026 for the three Spanish distribution networks.

| File | Distributor | CNMC Code | Territory |
|---|---|---|---|
| `R1-001_Demanda.csv` | **i-DE (Iberdrola)** | R1-001 | Centre, North, Aragon, Castilla, Valencia, Murcia |
| `R1-002_demanda.csv` | **Endesa** | R1-002 | Cataluña, Andalucía, Extremadura, Canarias |
| `R1299_demanda.csv` | **Viesgo (E.ON)** | R1-299 | Cantabria, Asturias |

**Key field:** `Capacidad firme disponible (MW)` — confirmed available headroom for new demand connections.

**Coordinate reference:** UTM ETRS89 zone 30N (EPSG:25830) → reprojected to WGS84. Points outside Spain bounding box [35.5–44.5°N, 9.5°W–4.5°E] are dropped as data artefacts.

**Assumption:** Spanish number format (comma decimal, dot thousands) is parsed before numeric conversion.

In [2]:
def load_grid_csv(filepath, distributor_name, crs='EPSG:25830'):
    """Load a CNMC grid capacity CSV and return a GeoDataFrame in WGS84."""
    df = pd.read_csv(filepath, sep=';')
    df.columns = [c.strip() for c in df.columns]

    # Flexible column detection handles suffix variants (e.g. 'UTM X [1]' in R1-002)
    x_col   = next((c for c in df.columns if 'UTM X' in c), None)
    y_col   = next((c for c in df.columns if 'UTM Y' in c), None)
    cap_col = next((c for c in df.columns if 'disponible' in c), None)

    for col in [c for c in [x_col, y_col, cap_col] if c]:
        df[col] = (
            df[col].astype(str).str.strip()
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .replace({'nan': np.nan, '': np.nan})
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=[x_col, y_col]).copy()
    df['capacity_available_mw'] = df[cap_col].fillna(0) if cap_col else 0.0
    df['distributor'] = distributor_name

    # Province column name varies across files
    prov_col = next((c for c in df.columns if c.strip().lower() in ['provincia', 'province']), None)
    df['province_raw'] = df[prov_col] if prov_col else ''

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[x_col], df[y_col]),
        crs=crs
    ).to_crs(4326)

    # Filter to mainland Spain + islands bounding box
    gdf = gdf[
        (gdf.geometry.y >= 35.5) & (gdf.geometry.y <= 44.5) &
        (gdf.geometry.x >= -9.5) & (gdf.geometry.x <=  4.5)
    ].copy()

    return gdf[['distributor', 'province_raw', 'capacity_available_mw', 'geometry']].reset_index(drop=True)


gdf_ide    = load_grid_csv(DATA / '2026_04_01_R1-001_Demanda.csv', 'i-DE')
gdf_endesa = load_grid_csv(DATA / '2026_04_01_R1-002_demanda.csv', 'Endesa')
gdf_viesgo = load_grid_csv(DATA / '2026_04_01_R1299_demanda.csv',  'Viesgo')

gdf_grid = pd.concat([gdf_ide, gdf_endesa, gdf_viesgo], ignore_index=True)

print(f'Grid nodes loaded:')
print(f'  i-DE    : {len(gdf_ide):>5,} nodes')
print(f'  Endesa  : {len(gdf_endesa):>5,} nodes')
print(f'  Viesgo  : {len(gdf_viesgo):>5,} nodes')
print(f'  Total   : {len(gdf_grid):>5,} nodes')
print()
print('Grid capacity summary (all distributors):')
print(gdf_grid.groupby('distributor')['capacity_available_mw'].agg(['count','sum','max','mean']).round(2).to_string())

Grid nodes loaded:
  i-DE    : 3,017 nodes
  Endesa  :   679 nodes
  Viesgo  : 1,793 nodes
  Total   : 5,489 nodes

Grid capacity summary (all distributors):
             count      sum     max  mean
distributor                              
Endesa         679   291.53   16.09  0.43
Viesgo        1793   659.29   55.96  0.37
i-DE          3017  3053.71  112.34  1.01


---
## 2. Existing EV Charger Baseline (DGT National Access Point)

The **DGT DATEX II v3 XML feed** is Spain's National Access Point for EV charging infrastructure — updated daily by registered operators under Royal Decree 569/2020.

**Filtering for baseline count:** Only stations with at least one connector rated ≥50 kW are counted as 'HPC-capable' interurban chargers. The DGT feed stores power in **Watts**; 50 kW = 50,000 W.

**Limitation:** The feed includes urban chargers (city streets, car parks). For `total_existing_stations_baseline` (File 1), we apply a spatial filter retaining only stations within 5 km of our defined TEN-T corridors — a defensible proxy for 'autopistas, autovías, and carreteras nacionales' as required by the datathon.

> Source: DGT DATEX II v3 — `infocar.dgt.es/datex2`

In [3]:
# ── XML helper functions (namespace-agnostic) ─────────────────────────────────
def strip_ns(tag):
    return tag.split('}', 1)[-1]

def find_child_text(elem, path_tags):
    cur = elem
    for tag in path_tags:
        nxt = next((c for c in cur if strip_ns(c.tag) == tag), None)
        if nxt is None:
            return None
        cur = nxt
    return (cur.text or '').strip() if cur.text else None

def extract_coordinates(site):
    lat = lon = None
    for el in site.iter():
        tag = strip_ns(el.tag)
        txt = (el.text or '').strip()
        if tag == 'latitude' and txt:
            try: lat = float(txt)
            except: pass
        elif tag == 'longitude' and txt:
            try: lon = float(txt)
            except: pass
    return lat, lon

# ── Fetch DGT DATEX II feed ───────────────────────────────────────────────────
print('Fetching DGT DATEX II EV charger feed...')
r = requests.get(
    'https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml',
    timeout=90
)
r.raise_for_status()
root = ET.fromstring(r.content)
print(f'XML loaded — root: {strip_ns(root.tag)}')

# ── Parse sites ───────────────────────────────────────────────────────────────
site_rows = []
sites_xml = [el for el in root.iter() if strip_ns(el.tag) == 'energyInfrastructureSite']
print(f'Sites in feed: {len(sites_xml):,}')

for site in sites_xml:
    lat, lon = extract_coordinates(site)
    if lat is None or lon is None:
        continue
    operator = find_child_text(site, ['operator', 'name', 'values', 'value'])
    # Collect max power across all connectors (values are in Watts per DATEX II spec)
    max_power_w = None
    n_connectors = 0
    for conn in site.iter():
        if strip_ns(conn.tag) != 'connector':
            continue
        n_connectors += 1
        pw_txt = find_child_text(conn, ['maxPowerAtSocket'])
        try:
            pw = float(pw_txt)
            max_power_w = max(max_power_w, pw) if max_power_w is not None else pw
        except (TypeError, ValueError):
            pass
    site_rows.append({
        'latitude': lat, 'longitude': lon,
        'operator': operator,
        'max_power_w': max_power_w,
        'n_connectors': n_connectors,
    })

sites_df = pd.DataFrame(site_rows)
sites_gdf = gpd.GeoDataFrame(
    sites_df,
    geometry=gpd.points_from_xy(sites_df['longitude'], sites_df['latitude']),
    crs=CRS_GEO
)

print(f'Sites with coordinates: {len(sites_gdf):,}')
print(f'HPC sites (≥50 kW):     {(sites_gdf["max_power_w"] >= HPC_MIN_POWER_W).sum():,}')
print()
print('Power tier breakdown:')
tiers = pd.cut(sites_gdf['max_power_w'].dropna(),
               bins=[0, 22000, 50000, 150000, float('inf')],
               labels=['<22kW (AC slow)', '22–50kW (AC fast)', '50–150kW (DC fast)', '>150kW (HPC ultra)'])
print(tiers.value_counts().sort_index().to_string())

Fetching DGT DATEX II EV charger feed...
XML loaded — root: payload
Sites in feed: 12,075
Sites with coordinates: 12,075
HPC sites (≥50 kW):     5,338

Power tier breakdown:
max_power_w
<22kW (AC slow)       5951
22–50kW (AC fast)     3500
50–150kW (DC fast)    2200
>150kW (HPC ultra)     424


---
## 3. Province EV Demand (from Notebook 2.2)

Loads the province-level EV fleet projections output by `Geographic_Demand_Distribution.ipynb` (Notebook 2.2).

These are used as a **secondary demand signal** to weight station sizing in high-demand provinces.

In [4]:
df_province = pd.read_csv(OUT / 'province_demand_2027.csv')

# Total EV fleet 2027 — from Notebook 2.1 (SARIMA forecast)
df_ev_total = pd.read_csv(OUT / 'total_ev_projected_2027.csv')
TOTAL_EV_2027 = int(df_ev_total['total_ev_projected_2027'].iloc[0])

print(f'Total EV fleet projected 2027: {TOTAL_EV_2027:,}')
print(f'Provinces loaded: {len(df_province)}')
print()
print('Top 10 provinces by EV fleet 2027:')
print(
    df_province[['province_code','province_name','ev_fleet_2027','share_pct']]
    .head(10).to_string(index=False)
)

Total EV fleet projected 2027: 1,412,640
Provinces loaded: 52

Top 10 provinces by EV fleet 2027:
province_code          province_name  ev_fleet_2027  share_pct
            M                 Madrid         635635    44.9963
            B              Barcelona         184440    13.0564
            V      Valencia/València          56301     3.9855
            A       Alicante/Alacant          45556     3.2249
           GC             Las Palmas          35456     2.5099
           IB          Illes Balears          33221     2.3517
           MA                 Málaga          31745     2.2472
           TF Santa Cruz de Tenerife          23926     1.6937
           BI                Bizkaia          22951     1.6247
           SE                Sevilla          22554     1.5966


---
## 4. TEN-T Corridor Definition & Candidate Site Sampling

### Corridor Selection
Spain's TEN-T Core and Comprehensive corridors are mapped to 12 strategic interurban axes. These align with the corridors mandated in AFIR Regulation 2023/1804 Annex II (TEN-T road network).

| Corridor | Route | Classification |
|---|---|---|
| A-1 | Madrid → Burgos → Bilbao | TEN-T Core (Atlantic) |
| A-2 | Madrid → Zaragoza → Barcelona | TEN-T Core (Mediterranean) |
| A-3 | Madrid → Valencia | TEN-T Comprehensive |
| A-4 | Madrid → Córdoba → Sevilla → Cádiz | TEN-T Core (Atlantic) |
| A-5 | Madrid → Mérida → Badajoz | TEN-T Comprehensive |
| A-6 | Madrid → Valladolid → A Coruña | TEN-T Comprehensive |
| AP-7 | French border → Barcelona → Algeciras | TEN-T Core (Mediterranean) |
| A-8 | Bilbao → Santander → A Coruña | TEN-T Comprehensive |
| A-66 | Oviedo → Salamanca → Mérida → Sevilla | TEN-T Comprehensive |
| A-23 | Sagunto → Zaragoza → France | TEN-T Comprehensive |
| A-45 | Córdoba → Málaga | TEN-T Comprehensive |
| A-92 | Sevilla → Granada → Almería | TEN-T Comprehensive |

### Sampling Method
Each corridor is defined as a polyline (sequence of GPS waypoints) representing the road geometry. Points are interpolated every **60 km** (AFIR Core spacing) along the projected metric line (EPSG:25830), then back-transformed to WGS84.

**Assumption:** The 60 km sampling interval is used uniformly across all corridors. For Comprehensive corridors (where AFIR allows 100 km), this is conservative — it generates more candidates than strictly required, but the AFIR gap filter in Section 4 will remove candidates already served by existing chargers.

In [5]:
# ── TEN-T corridor waypoints (lon, lat) in WGS84 ─────────────────────────────
# Each tuple is an approximate GPS waypoint along the road centreline
CORRIDORS = {
    'A-1':  [(-3.68,40.45),(-3.41,40.94),(-3.68,41.67),(-3.68,42.35),(-2.94,42.68),(-2.68,42.85),(-2.93,43.26)],
    'A-2':  [(-3.50,40.47),(-3.16,40.63),(-2.43,41.08),(-1.64,41.35),(-0.88,41.65),(0.62,41.62),(2.10,41.38)],
    'A-3':  [(-3.62,40.40),(-3.00,40.01),(-2.43,39.76),(-1.89,39.56),(-1.09,39.49),(-0.38,39.46)],
    'A-4':  [(-3.65,40.37),(-3.60,40.04),(-3.38,39.53),(-3.37,38.99),(-3.77,38.09),(-4.78,37.88),(-5.96,37.38),(-6.12,36.68),(-6.29,36.52)],
    'A-5':  [(-3.79,40.37),(-4.83,39.96),(-5.53,39.89),(-6.35,38.91),(-6.97,38.88)],
    'A-6':  [(-3.87,40.47),(-4.52,41.13),(-4.72,41.65),(-5.68,42.00),(-6.59,42.55),(-7.56,43.01),(-8.41,43.36)],
    'AP-7': [(2.87,42.42),(2.82,41.98),(2.17,41.39),(1.25,41.12),(0.00,40.63),(-0.38,39.46),(-0.48,38.35),(-1.13,37.98),(-2.46,36.84),(-4.42,36.72),(-5.45,36.13)],
    'A-8':  [(-2.92,43.26),(-3.81,43.46),(-4.50,43.54),(-5.84,43.36),(-7.00,43.37),(-7.56,43.01),(-8.41,43.36)],
    'A-66': [(-5.84,43.36),(-5.77,43.25),(-5.68,42.70),(-5.68,42.00),(-5.66,40.97),(-6.12,39.47),(-6.35,38.91),(-5.96,37.38)],
    'A-23': [(-0.27,39.68),(-1.11,40.35),(-0.88,41.65),(-0.41,42.14),(0.20,42.79)],
    'A-45': [(-4.78,37.88),(-4.49,37.41),(-4.56,37.02),(-4.42,36.72)],
    'A-92': [(-5.96,37.38),(-5.10,37.23),(-4.56,37.02),(-3.60,37.18),(-3.13,37.30),(-2.46,36.84)],
}

# ── Coordinate transformers ───────────────────────────────────────────────────
to_metric   = pyproj.Transformer.from_crs('EPSG:4326', CRS_METRIC, always_xy=True)
to_geo      = pyproj.Transformer.from_crs(CRS_METRIC, 'EPSG:4326', always_xy=True)

# ── Sample candidate points every AFIR_SPACING_KM along each corridor ─────────
candidates = []

for route, waypoints in CORRIDORS.items():
    # Project waypoints to metric CRS
    proj_pts = [to_metric.transform(lon, lat) for lon, lat in waypoints]
    line_m   = LineString(proj_pts)
    total_m  = line_m.length

    # Sample every 60 km (inclusive of start and end)
    spacing_m = AFIR_SPACING_KM * 1000
    n_pts     = int(total_m // spacing_m) + 1

    for i in range(n_pts + 1):
        dist   = min(i * spacing_m, total_m)
        pt_m   = line_m.interpolate(dist)
        lon, lat = to_geo.transform(pt_m.x, pt_m.y)
        candidates.append({
            'route_segment': route,
            'latitude':      round(lat, 6),
            'longitude':     round(lon, 6),
            'corridor_len_km': round(total_m / 1000, 1),
        })

cand_gdf = gpd.GeoDataFrame(
    candidates,
    geometry=gpd.points_from_xy(
        [c['longitude'] for c in candidates],
        [c['latitude']  for c in candidates]
    ),
    crs=CRS_GEO
)

print(f'Total candidate sites sampled: {len(cand_gdf)}')
print()
print('Candidates per corridor:')
summary = cand_gdf.groupby('route_segment').agg(
    n_candidates=('latitude','count'),
    corridor_km=('corridor_len_km','first')
).reset_index()
print(summary.to_string(index=False))

Total candidate sites sampled: 114

Candidates per corridor:
route_segment  n_candidates  corridor_km
          A-1             8        367.9
          A-2            10        509.1
         A-23             8        403.9
          A-3             7        306.1
          A-4            11        574.4
         A-45             4        137.4
          A-5             7        343.6
          A-6            10        511.1
         A-66            13        677.0
          A-8             9        476.4
         A-92             7        340.7
         AP-7            20       1125.0


---
## 5. AFIR Gap Analysis

For each candidate point, we compute the **distance to the nearest existing HPC charger** (≥50 kW) using a KD-tree spatial index on the projected coordinates (EPSG:25830).

A candidate becomes a **proposed station** if:
> distance to nearest HPC charger > **30 km** (= AFIR spacing ÷ 2)

This ensures the new station closes the gap to within AFIR compliance from both sides.

**HPC charger definition:** Site with at least one connector rated ≥50,000 W (=50 kW) per DGT DATEX II. This threshold includes DC fast chargers capable of meaningful highway use.

**Assumption:** Any charger within 30 km of a candidate is considered to provide AFIR coverage at that point. This is conservative — in practice, the charger must be on-route (not off a motorway junction), but junction distances are typically <3 km and do not materially affect the 30 km gap threshold.

In [6]:
# ── Filter existing sites to HPC only ────────────────────────────────────────
hpc_gdf = sites_gdf[sites_gdf['max_power_w'] >= HPC_MIN_POWER_W].copy()
print(f'HPC sites used for gap analysis: {len(hpc_gdf):,}')

# ── Project both layers to metric CRS for accurate distance calculations ──────
hpc_metric  = hpc_gdf.to_crs(CRS_METRIC)
cand_metric = cand_gdf.to_crs(CRS_METRIC)

# ── Build KD-tree on HPC charger coordinates ──────────────────────────────────
hpc_coords = np.column_stack([
    hpc_metric.geometry.x,
    hpc_metric.geometry.y
])
hpc_tree = KDTree(hpc_coords)

# ── Query: nearest HPC distance for each candidate ────────────────────────────
cand_coords = np.column_stack([
    cand_metric.geometry.x,
    cand_metric.geometry.y
])
dist_m, _ = hpc_tree.query(cand_coords, k=1)
dist_km   = dist_m / 1000

cand_gdf['nearest_hpc_km'] = dist_km.round(2)

# ── Retain only AFIR gaps ─────────────────────────────────────────────────────
proposed_gdf = cand_gdf[cand_gdf['nearest_hpc_km'] > GAP_RADIUS_KM].copy()
proposed_gdf = proposed_gdf.reset_index(drop=True)

print(f'\nAFIR gap analysis results:')
print(f'  Candidates evaluated  : {len(cand_gdf):>4}')
print(f'  Already served (<30km): {(cand_gdf["nearest_hpc_km"] <= GAP_RADIUS_KM).sum():>4}')
print(f'  AFIR gaps → proposed  : {len(proposed_gdf):>4}')
print()
print('Proposed stations per corridor:')
print(
    proposed_gdf.groupby('route_segment').size()
    .reset_index(name='n_proposed').sort_values('n_proposed', ascending=False)
    .to_string(index=False)
)

HPC sites used for gap analysis: 5,338

AFIR gap analysis results:
  Candidates evaluated  :  114
  Already served (<30km):  113
  AFIR gaps → proposed  :    1

Proposed stations per corridor:
route_segment  n_proposed
         A-23           1


---
## 6. Traffic Demand & Station Sizing

### Traffic Data Source
Daily vehicle flows per corridor are fetched from the **Ministry of Transport BigData Movilidad 2 ArcGIS REST API**, filtered to our 12 corridor road names. This API covers the entire Spanish road network with segment-level traveller counts.

For each proposed station, the **median daily traffic on its corridor** is used as the demand signal.

### n_chargers Formula
$$n_{chargers} = \left\lceil \frac{traffic_{daily} \times p_{EV} \times p_{stop} \times t_{session}}{H_{operational} \times u_{target}} \right\rceil$$

| Parameter | Value | Source |
|---|---|---|
| $p_{EV}$ — EV share of fleet 2027 | 3.8% | DGT + Notebook 2.1 SARIMA forecast |
| $p_{stop}$ — P(EV stops to charge on interurban trip) | 15% | MITMA 2023 mobility survey |
| $t_{session}$ — charging session duration (h) | 0.33 h (20 min) | 50 kWh ÷ 150 kW |
| $H_{operational}$ — daily service hours | 16 h | 06:00–22:00 operational window |
| $u_{target}$ — charger utilisation target | 75% | McKinsey EV Infrastructure 2023 |

**Result clamped to [2, 12]:** minimum 2 per AFIR mandate; maximum 12 per space and cost constraints.

In [7]:
# ── Fetch corridor traffic from Ministry of Transport ArcGIS ──────────────────
# We filter server-side to just our 12 corridor names for speed
MOT_URL = 'https://mapas.fomento.gob.es/arcgis2/rest/services/BigData/Movilidad_Big_Data_2/MapServer/1282/query'

corridor_names = list(CORRIDORS.keys())
# Build SQL IN clause — 'nombre' field stores the road code in this layer
in_clause = ','.join([f"'{c}'" for c in corridor_names])
where = f"nombre IN ({in_clause})"

traffic_rows = []
offset       = 0
BATCH        = 2000

print(f'Fetching traffic for corridors: {corridor_names}')
while True:
    params = {
        'f': 'json', 'where': where,
        'outFields': 'nombre,Total',
        'returnGeometry': 'false',
        'resultOffset': offset,
        'resultRecordCount': BATCH
    }
    try:
        resp = requests.get(MOT_URL, params=params, timeout=60)
        resp.raise_for_status()
        features = resp.json().get('features', [])
    except Exception as e:
        print(f'  Batch {offset}: {e}')
        break

    for f in features:
        a = f.get('attributes', {})
        traffic_rows.append({'route_segment': (a.get('nombre') or '').strip().upper(),
                             'Total': a.get('Total')})
    if len(features) < BATCH:
        break
    offset += BATCH
    time.sleep(0.1)

traffic_df = pd.DataFrame(traffic_rows)
traffic_df['Total'] = pd.to_numeric(traffic_df['Total'], errors='coerce')

# Median daily vehicles per corridor (robust to segment-level noise)
corridor_traffic = (
    traffic_df[traffic_df['Total'].notna() & (traffic_df['Total'] > 0)]
    .groupby('route_segment')['Total']
    .median()
    .reset_index()
    .rename(columns={'Total': 'median_daily_traffic'})
)

# Fallback: corridors not returned by API get a conservative 10,000 vehicles/day
for route in corridor_names:
    if route not in corridor_traffic['route_segment'].values:
        corridor_traffic = pd.concat([
            corridor_traffic,
            pd.DataFrame([{'route_segment': route, 'median_daily_traffic': 10_000}])
        ], ignore_index=True)

print(f'\nTraffic records fetched: {len(traffic_rows):,}')
print()
print('Median daily traffic per corridor (vehicles/day):')
print(corridor_traffic.sort_values('median_daily_traffic', ascending=False).to_string(index=False))

Fetching traffic for corridors: ['A-1', 'A-2', 'A-3', 'A-4', 'A-5', 'A-6', 'AP-7', 'A-8', 'A-66', 'A-23', 'A-45', 'A-92']

Traffic records fetched: 35,036

Median daily traffic per corridor (vehicles/day):
route_segment  median_daily_traffic
          A-8              7852.810
         A-45              7527.700
          A-1              7489.495
          A-2              6499.600
          A-3              6257.205
          A-5              5833.585
         A-66              5566.020
         A-23              5146.325
          A-6              5071.980
         A-92              4727.100
         AP-7              1996.695
          A-4              1707.890


In [8]:
# ── Merge traffic into proposed stations ──────────────────────────────────────
proposed_gdf = proposed_gdf.merge(corridor_traffic, on='route_segment', how='left')
proposed_gdf['median_daily_traffic'] = proposed_gdf['median_daily_traffic'].fillna(10_000)

# ── n_chargers formula ────────────────────────────────────────────────────────
def calc_n_chargers(daily_traffic):
    ev_daily    = daily_traffic * EV_SHARE_2027
    stop_demand = ev_daily * STOP_PROBABILITY                   # EVs needing charge at this station
    session_h   = stop_demand * SESSION_DURATION_H              # charger-hours needed per day
    chargers    = session_h / (OPERATIONAL_HOURS * UTILIZATION_TARGET)
    return int(min(MAX_CHARGERS, max(MIN_CHARGERS, math.ceil(chargers))))

proposed_gdf['n_chargers_proposed'] = proposed_gdf['median_daily_traffic'].apply(calc_n_chargers)
proposed_gdf['estimated_demand_kw'] = proposed_gdf['n_chargers_proposed'] * CHARGER_KW

print('Station sizing summary:')
print(proposed_gdf.groupby('route_segment')[['n_chargers_proposed', 'estimated_demand_kw']]
      .agg(['mean','min','max']).round(1).to_string())
print()
print('n_chargers distribution:')
print(proposed_gdf['n_chargers_proposed'].value_counts().sort_index().to_string())

Station sizing summary:
              n_chargers_proposed         estimated_demand_kw          
                             mean min max                mean  min  max
route_segment                                                          
A-23                          2.0   2   2               300.0  300  300

n_chargers distribution:
n_chargers_proposed
2    1


---
## 7. Grid Status Assignment

For each proposed station, we identify the **nearest substation** across all three distributors using a KD-tree on projected coordinates (EPSG:25830).

### Grid Status Thresholds

| Status | Available capacity | Justification |
|---|---|---|
| **Sufficient** | ≥ 5 MW | Accommodates up to 33 chargers at 150 kW with headroom; no immediate reinforcement needed |
| **Moderate** | 1 – 5 MW | Covers a standard 6-charger station (0.9 MW demand); feasible but requires coordination with distributor |
| **Congested** | < 1 MW | Total headroom below one standard station's demand; grid reinforcement required before connection |

Thresholds are anchored to the 150 kW standard charger power (fixed by datathon rules). A 6-charger station at full capacity draws 900 kW = 0.9 MW; we set the Moderate/Congested boundary at 1 MW to include a small safety margin.

### Distributor Network Assignment
The `distributor_network` field (required in File 3) is taken from the distributor of the nearest matched substation. Where multiple substations from different distributors are equidistant, the one with highest available capacity is preferred.

In [9]:
# ── Project grid nodes to metric CRS ─────────────────────────────────────────
grid_metric = gdf_grid.to_crs(CRS_METRIC).copy()
grid_coords = np.column_stack([grid_metric.geometry.x, grid_metric.geometry.y])
grid_tree   = KDTree(grid_coords)

# ── Project proposed stations to metric CRS ───────────────────────────────────
prop_metric = proposed_gdf.to_crs(CRS_METRIC).copy()
prop_coords = np.column_stack([prop_metric.geometry.x, prop_metric.geometry.y])

# ── Nearest-substation lookup ─────────────────────────────────────────────────
dist_grid_m, idx_grid = grid_tree.query(prop_coords, k=1)

proposed_gdf['nearest_substation_km']  = (dist_grid_m / 1000).round(2)
proposed_gdf['capacity_available_mw']  = gdf_grid.iloc[idx_grid]['capacity_available_mw'].values
proposed_gdf['distributor_network']    = gdf_grid.iloc[idx_grid]['distributor'].values

# ── Classify grid_status ──────────────────────────────────────────────────────
def classify_grid(mw):
    if mw >= GRID_SUFFICIENT_MW:
        return 'Sufficient'
    elif mw >= GRID_MODERATE_MW:
        return 'Moderate'
    else:
        return 'Congested'

proposed_gdf['grid_status'] = proposed_gdf['capacity_available_mw'].apply(classify_grid)

print('Grid status distribution:')
print(proposed_gdf['grid_status'].value_counts().to_string())
print()
print('Distributor network distribution:')
print(proposed_gdf['distributor_network'].value_counts().to_string())
print()
print('Grid status × Distributor:')
print(pd.crosstab(proposed_gdf['distributor_network'], proposed_gdf['grid_status']).to_string())

Grid status distribution:
grid_status
Congested    1

Distributor network distribution:
distributor_network
Viesgo    1

Grid status × Distributor:
grid_status          Congested
distributor_network           
Viesgo                       1


---
## 8. File 2 — Proposed Charging Locations

**File 2** lists every new charging station proposed by the team.

Field definitions (per datathon specification):
- `location_id` — sequential identifier `IBE_001`, `IBE_002`, …
- `latitude` / `longitude` — WGS84 decimal degrees
- `route_segment` — official road designation (A-3, AP-7, …)
- `n_chargers_proposed` — sized by demand model (Section 6)
- `estimated_demand_kw` — n_chargers × 150 kW (Rule 2: fixed 150 kW per charger)
- `grid_status` — Sufficient / Moderate / Congested (Rule 1: based on nearest substation available MW)

In [10]:
# ── Assign sequential location IDs ───────────────────────────────────────────
proposed_gdf = proposed_gdf.reset_index(drop=True)
proposed_gdf['location_id'] = [f'IBE_{i+1:03d}' for i in range(len(proposed_gdf))]

# ── Build File 2 in exact required column order ───────────────────────────────
file2 = proposed_gdf[[
    'location_id', 'latitude', 'longitude',
    'route_segment', 'n_chargers_proposed',
    'estimated_demand_kw', 'grid_status'
]].copy()

file2.to_csv(OUT / 'File 2.csv', index=False)
print(f'File 2 saved → outputs/File 2.csv')
print(f'Rows: {len(file2)}')
print()
print('File 2 — structure and format:')
print(file2.dtypes.to_string())
print()
print('File 2 — first 10 rows:')
print(file2.head(10).to_string(index=False))

File 2 saved → outputs/File 2.csv
Rows: 1

File 2 — structure and format:
location_id             object
latitude               float64
longitude              float64
route_segment           object
n_chargers_proposed      int64
estimated_demand_kw      int64
grid_status             object

File 2 — first 10 rows:
location_id  latitude  longitude route_segment  n_chargers_proposed  estimated_demand_kw grid_status
    IBE_001     42.79        0.2          A-23                    2                  300   Congested


---
## 9. File 3 — Friction Points (Moderate + Congested only)

**File 3** logs locations where EV charging demand exists but the grid requires reinforcement — "friction points".

**Rule 3 (mandatory):** Only stations with `grid_status ∈ {Moderate, Congested}` from File 2 are included. `Sufficient` stations are excluded.

Field definitions:
- `bottleneck_id` — sequential `FRIC_001`, `FRIC_002`, …
- `distributor_network` — i-DE, Endesa, or Viesgo (from nearest substation)
- `estimated_demand_kw` — n_chargers × 150 kW (Rule 2)
- `grid_status` — Moderate or Congested only

In [11]:
# Rule 3: only Moderate or Congested locations
file3_src = proposed_gdf[proposed_gdf['grid_status'].isin(['Moderate', 'Congested'])].copy()
file3_src = file3_src.reset_index(drop=True)
file3_src['bottleneck_id'] = [f'FRIC_{i+1:03d}' for i in range(len(file3_src))]

file3 = file3_src[[
    'bottleneck_id', 'latitude', 'longitude',
    'route_segment', 'distributor_network',
    'estimated_demand_kw', 'grid_status'
]].copy()

file3.to_csv(OUT / 'File 3.csv', index=False)
print(f'File 3 saved → outputs/File 3.csv')
print(f'Rows: {len(file3)}')
print()
print('File 3 — structure and format:')
print(file3.dtypes.to_string())
print()
print('File 3 — first 10 rows:')
print(file3.head(10).to_string(index=False))
print()
print('Friction point breakdown by distributor and status:')
print(pd.crosstab(file3['distributor_network'], file3['grid_status']).to_string())

File 3 saved → outputs/File 3.csv
Rows: 1

File 3 — structure and format:
bottleneck_id           object
latitude               float64
longitude              float64
route_segment           object
distributor_network     object
estimated_demand_kw      int64
grid_status             object

File 3 — first 10 rows:
bottleneck_id  latitude  longitude route_segment distributor_network  estimated_demand_kw grid_status
     FRIC_001     42.79        0.2          A-23              Viesgo                  300   Congested

Friction point breakdown by distributor and status:
grid_status          Congested
distributor_network           
Viesgo                       1


---
## 10. File 1 — Global Network KPIs (Scorecard)

A single-row summary capturing the four required KPI fields.

| Field | Source |
|---|---|
| `total_proposed_stations` | Row count of File 2 |
| `total_existing_stations_baseline` | DGT DATEX II sites with ≥50 kW within 5 km of our TEN-T corridors |
| `total_friction_points` | Row count of File 3 |
| `total_ev_projected_2027` | SARIMA forecast output — Notebook 2.1 |

**Methodology note (baseline count):** The datathon specifies filtering the NAP dataset to "autopistas, autovías, and carreteras nacionales". We implement this spatially: any site within **5 km** of our TEN-T corridor polylines is counted as an interurban charger. This is more robust than address-string matching (which is inconsistent in the DGT feed) and directly reflects geographic proximity to the strategic road network.

In [12]:
# ── Build corridor union geometry for spatial filtering ───────────────────────
corridor_lines = []
for route, waypoints in CORRIDORS.items():
    proj_pts = [to_metric.transform(lon, lat) for lon, lat in waypoints]
    corridor_lines.append(LineString(proj_pts))

from shapely.ops import unary_union
corridor_union_m = unary_union(corridor_lines)

# ── Count existing sites within 5 km of any corridor ─────────────────────────
sites_metric = sites_gdf.to_crs(CRS_METRIC).copy()
baseline_mask = sites_metric.geometry.apply(
    lambda pt: corridor_union_m.distance(pt) <= 5_000
)
total_existing_baseline = int(baseline_mask.sum())

# ── Build File 1 ──────────────────────────────────────────────────────────────
file1 = pd.DataFrame([{
    'total_proposed_stations':        len(file2),
    'total_existing_stations_baseline': total_existing_baseline,
    'total_friction_points':          len(file3),
    'total_ev_projected_2027':        TOTAL_EV_2027,
}])

file1.to_csv(OUT / 'File 1.csv', index=False)
print('File 1 saved → outputs/File 1.csv')
print()
print('File 1 — structure and format:')
print(file1.dtypes.to_string())
print()
print('File 1 — Global Network KPIs:')
print(file1.T.to_string(header=False))

File 1 saved → outputs/File 1.csv

File 1 — structure and format:
total_proposed_stations             int64
total_existing_stations_baseline    int64
total_friction_points               int64
total_ev_projected_2027             int64

File 1 — Global Network KPIs:
total_proposed_stations                 1
total_existing_stations_baseline     5268
total_friction_points                   1
total_ev_projected_2027           1412640


---
## 11. BI Visualisation — Proposed Charging Station Map

An interactive Folium map meeting all datathon BI requirements:

**Mandatory layer (minimum requirement):**
- All proposed stations from File 2, colour-coded by `grid_status`:
  - 🟢 **Green** — Sufficient
  - 🟡 **Yellow** — Moderate  
  - 🔴 **Red** — Congested

**Additional layers (above baseline):**
- Existing HPC charger density (heatmap)
- TEN-T corridor polylines colour-coded by distributor territory
- Popup with full station detail on click (location_id, route, n_chargers, demand_kw, grid_status)

**Format:** Self-contained HTML — no login, no installation, no external dependencies at runtime.

In [13]:
from folium.plugins import HeatMap

STATUS_COLOR = {'Sufficient': '#2ca02c', 'Moderate': '#f0a500', 'Congested': '#d62728'}
DISTRIB_COLOR = {'A-1':'#1f77b4','A-2':'#1f77b4','A-3':'#ff7f0e','A-4':'#1f77b4',
                 'A-5':'#ff7f0e','A-6':'#ff7f0e','AP-7':'#2ca02c','A-8':'#9467bd',
                 'A-66':'#9467bd','A-23':'#8c564b','A-45':'#e377c2','A-92':'#17becf'}

m_bi = folium.Map(location=SPAIN_CENTER, zoom_start=6, tiles='CartoDB positron')

# ── Layer 1: TEN-T corridor polylines ─────────────────────────────────────────
fg_corridors = folium.FeatureGroup(name='TEN-T Corridors', show=True)
for route, waypoints in CORRIDORS.items():
    coords = [(lat, lon) for lon, lat in waypoints]
    folium.PolyLine(
        coords, color=DISTRIB_COLOR.get(route, '#888888'),
        weight=2, opacity=0.5, tooltip=route
    ).add_to(fg_corridors)
fg_corridors.add_to(m_bi)

# ── Layer 2: Existing HPC charger density ─────────────────────────────────────
fg_hpc = folium.FeatureGroup(name='Existing HPC Chargers (density)', show=True)
heat_data = [
    [row.geometry.y, row.geometry.x]
    for _, row in hpc_gdf.iterrows()
]
HeatMap(heat_data, radius=10, blur=12, min_opacity=0.3).add_to(fg_hpc)
fg_hpc.add_to(m_bi)

# ── Layer 3: Proposed stations (colour-coded by grid_status) ──────────────────
fg_sufficient = folium.FeatureGroup(name='Proposed — Sufficient', show=True)
fg_moderate   = folium.FeatureGroup(name='Proposed — Moderate',   show=True)
fg_congested  = folium.FeatureGroup(name='Proposed — Congested',  show=True)

layer_map = {'Sufficient': fg_sufficient, 'Moderate': fg_moderate, 'Congested': fg_congested}

for _, row in proposed_gdf.iterrows():
    status = row['grid_status']
    color  = STATUS_COLOR[status]
    popup_html = (
        f"<b>{row['location_id']}</b><br>"
        f"Route: <b>{row['route_segment']}</b><br>"
        f"Chargers: <b>{row['n_chargers_proposed']}</b> × 150 kW<br>"
        f"Demand: <b>{row['estimated_demand_kw']:,} kW</b><br>"
        f"Grid status: <b style='color:{color}'>{status}</b><br>"
        f"Distributor: {row['distributor_network']}<br>"
        f"Available capacity: {row['capacity_available_mw']:.1f} MW<br>"
        f"Nearest HPC: {row['nearest_hpc_km']:.1f} km"
    )
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=9,
        color='white', weight=1.5,
        fill=True, fill_color=color, fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"{row['location_id']} | {row['route_segment']} | {status}"
    ).add_to(layer_map[status])

for fg in [fg_sufficient, fg_moderate, fg_congested]:
    fg.add_to(m_bi)

# ── Layer control + legend ────────────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(m_bi)

n_suf = (proposed_gdf['grid_status']=='Sufficient').sum()
n_mod = (proposed_gdf['grid_status']=='Moderate').sum()
n_con = (proposed_gdf['grid_status']=='Congested').sum()

legend_html = (
    '<div style="position:fixed;bottom:35px;left:35px;z-index:9999;background:white;'
    'padding:14px 18px;border-radius:8px;border:1px solid #ccc;font-size:12px;line-height:2">'
    '<b style="font-size:13px">Proposed Stations</b><br>'
    f'<span style="color:#2ca02c;font-size:16px">&#9679;</span> Sufficient &nbsp;({n_suf})<br>'
    f'<span style="color:#f0a500;font-size:16px">&#9679;</span> Moderate &nbsp;&nbsp;({n_mod})<br>'
    f'<span style="color:#d62728;font-size:16px">&#9679;</span> Congested &nbsp;({n_con})<br>'
    f'<hr style="margin:4px 0"><small>Total: <b>{len(proposed_gdf)}</b> stations</small>'
    '</div>'
)
m_bi.get_root().html.add_child(folium.Element(legend_html))

bi_path = str(OUT / 'File_BI_visualization.html')
m_bi.save(bi_path)
print(f'BI map saved → {bi_path}')
print(f'Open in browser: notebooks/outputs/File_BI_visualization.html')
IFrame(src=bi_path, width='100%', height=600)

BI map saved → outputs\File_BI_visualization.html
Open in browser: notebooks/outputs/File_BI_visualization.html


---
## 12. Output Summary & Verification

Final check: print structure and counts for all three output files.

In [14]:
print('=' * 60)
print('OUTPUT FILE VERIFICATION')
print('=' * 60)

for fname in ['File 1.csv', 'File 2.csv', 'File 3.csv']:
    df_check = pd.read_csv(OUT / fname)
    print(f'\n{fname}  ({len(df_check)} rows × {len(df_check.columns)} cols)')
    print(f'  Columns: {list(df_check.columns)}')
    print(f'  dtypes :')
    for col, dtype in df_check.dtypes.items():
        print(f'    {col:<35} {dtype}')
    if len(df_check) <= 5:
        print('  Data:')
        print(df_check.to_string(index=False))
    else:
        print(f'  Head(3):')
        print(df_check.head(3).to_string(index=False))

print('\n' + '=' * 60)
print('GLOBAL KPI SUMMARY')
print('=' * 60)
kpi = pd.read_csv(OUT / 'File 1.csv').iloc[0]
print(f'  Proposed stations          : {int(kpi["total_proposed_stations"]):,}')
print(f'  Existing baseline (NAP)    : {int(kpi["total_existing_stations_baseline"]):,}')
print(f'  Friction points            : {int(kpi["total_friction_points"]):,}')
print(f'  EV fleet projected 2027    : {int(kpi["total_ev_projected_2027"]):,}')
print(f'  Total new charging points  : {int(file2["n_chargers_proposed"].sum()):,}')
print(f'  Total new capacity (MW)    : {file2["estimated_demand_kw"].sum()/1000:.1f}')

OUTPUT FILE VERIFICATION

File 1.csv  (1 rows × 4 cols)
  Columns: ['total_proposed_stations', 'total_existing_stations_baseline', 'total_friction_points', 'total_ev_projected_2027']
  dtypes :
    total_proposed_stations             int64
    total_existing_stations_baseline    int64
    total_friction_points               int64
    total_ev_projected_2027             int64
  Data:
 total_proposed_stations  total_existing_stations_baseline  total_friction_points  total_ev_projected_2027
                       1                              5268                      1                  1412640

File 2.csv  (1 rows × 7 cols)
  Columns: ['location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed', 'estimated_demand_kw', 'grid_status']
  dtypes :
    location_id                         object
    latitude                            float64
    longitude                           float64
    route_segment                       object
    n_chargers_proposed               